In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ── output file name ──────────────────────────────────────────────────────────
OUTPUT_CSV = "au_results_no_temporal.csv"

EXPERIMENT_DIRS = {
    "GAUGE":                     "Experiments/20260603_213649_gauge_only_notemporal",
    "GAUGE + RADAR":             "Experiments/20260604_003951_gauge_radar_notemporal",
    "GAUGE + RADAR + SATELLITE": "Experiments/20260604_003951_gauge_radar_satellite_notemporal",
}

# metric key -> display label (column order preserved)
METRICS = {
    "station_median_rmse": "Station Median RMSE",
    "station_median_r":    "Station Median r",
    "precision":           "Precision",
    "recall":              "Recall",
    "f1":                  "F1",
    "pearson_r":           "Global r",
    "rmse":                "Global RMSE",
    "timestep_rmse":       "Timestep RMSE",
}
LABELS = list(METRICS.values())
ROUND = 4

def load_folds(exp_dir):
    return json.loads((Path(exp_dir) / "test_metrics_all_folds.json").read_text())

summary_rows = {}
per_fold_blocks = {}

for exp_name, exp_dir in EXPERIMENT_DIRS.items():
    try:
        folds = load_folds(exp_dir)
    except FileNotFoundError:
        print(f"⚠️  JSON not found (run unfinished?): {exp_dir}")
        continue

    block = pd.DataFrame(
        {label: [round(float(f[key]), ROUND) for f in folds]
         for key, label in METRICS.items()},
        index=pd.Index([f"Fold {i+1}" for i in range(len(folds))], name="Fold"),
    )
    per_fold_blocks[exp_name] = block

    summary_rows[exp_name] = {
        label: f"{np.nanmean(block[label]):.{ROUND}f} ± {np.nanstd(block[label]):.{ROUND}f}"
        for label in LABELS
    }

summary_df = pd.DataFrame.from_dict(summary_rows, orient="index")[LABELS]
summary_df.index.name = "Experiment"

# ── write summary + per-fold into ONE csv ─────────────────────────────────────
with open(OUTPUT_CSV, "w", encoding="utf-8") as fh:
    fh.write("SUMMARY (mean ± std across folds)\n")
    summary_df.to_csv(fh)
    fh.write("\n\nPER-FOLD RESULTS\n\n")
    for exp_name, block in per_fold_blocks.items():
        fh.write(f"{exp_name}\n")
        block.to_csv(fh)
        fh.write("\n")

In [1]:
import summarize_runs

In [9]:
!python3 summarize_runs.py --filter sumaggr --sort station_median_rmse


WAGGA RUN SUMMARY  (filter='sumaggr', sorted by station_median_rmse)
                              tag  folds   pearson_r station_median_r station_median_rmse          f1                flag
wagga_radar_satellite_log_sumaggr      5 0.229±0.179      0.245±0.203         0.693±0.082 0.213±0.228 COLLAPSE? high-var 

  REFERENCE  IDW          : pearson=0.690  smed_r=0.685  smed_RMSE=0.522  f1=0.668
  REFERENCE  gauge MSE-log : pearson=0.610  smed_r=0.672  smed_RMSE=0.569  f1=0.641

----------------------------------------------------------------------------------------------------
QUICK READS
----------------------------------------------------------------------------------------------------
No run beats IDW on both pearson r and station-median RMSE yet.

Possible collapses (check loss curve / scatter):
   ✗ wagga_radar_satellite_log_sumaggr        pearson=0.229  f1=0.213

NOTE: runs with folds<5 are partial — means will shift as more land.



In [1]:
!python3 summarize_runs.py --filter gate --sort station_median_rmse


WAGGA RUN SUMMARY  (filter='gate', sorted by station_median_rmse)
                                  tag  folds   pearson_r station_median_r station_median_rmse          f1 flag
gauge_radar_satellite_notemporal_gate      1 0.766±0.000      0.848±0.000         0.717±0.000 0.694±0.000     

  REFERENCE  IDW          : pearson=0.690  smed_r=0.685  smed_RMSE=0.522  f1=0.668
  REFERENCE  gauge MSE-log : pearson=0.610  smed_r=0.672  smed_RMSE=0.569  f1=0.641

----------------------------------------------------------------------------------------------------
QUICK READS
----------------------------------------------------------------------------------------------------
No run beats IDW on both pearson r and station-median RMSE yet.

NOTE: runs with folds<5 are partial — means will shift as more land.

